# Lab 7: Information Extraction & NER

**Name:** Nauman Ejaz  
**ID:** 04072212042  

This notebook extracts useful information from text, such as dates, amounts, invoice numbers, names, organizations, and locations.


## Setup

Run these commands only once if spaCy is not installed:

```python
pip install spacy
python -m spacy download en_core_web_sm
```


In [ ]:
import re
import json

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    print("spaCy is ready.")
except Exception:
    nlp = None
    print("spaCy model is not available. Regex parts will still work.")


## Part 1: Regular Expressions

These functions use simple regex patterns to extract dates, money amounts, and invoice/order numbers.


In [ ]:
def extract_dates(text):
    """
    Extract dates in common formats:
    MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY, and YYYY-MM-DD
    """
    patterns = [
        r"\d{1,2}/\d{1,2}/\d{4}",
        r"\d{1,2}-\d{1,2}-\d{4}",
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}",
        r"\d{4}-\d{2}-\d{2}"
    ]

    dates = []
    for pattern in patterns:
        dates.extend(re.findall(pattern, text, re.IGNORECASE))
    return dates


text = "Invoice date: 03/15/2024. Due: March 30, 2024."
print(extract_dates(text))


In [ ]:
def extract_amounts(text):
    """
    Extract currency amounts like:
    $1,250.50, 1250.50, and $1250
    """
    pattern = r"\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?|\$?\d+(?:\.\d{2})?"
    amounts = re.findall(pattern, text)

    cleaned = []
    for amount in amounts:
        amount = amount.replace("$", "").replace(",", "")
        cleaned.append(float(amount))
    return cleaned


text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45."
print(extract_amounts(text))


In [ ]:
def extract_invoice_number(text):
    """
    Extract invoice or order numbers.
    Examples:
    INV-2024-001, #12345, ORDER-ABC123
    """
    patterns = [
        r"INV-\d{4}-\d{3}",
        r"#\d{5,}",
        r"ORDER-[A-Z0-9]+",
        r"Invoice (?:Number|#):?\s*([A-Z0-9-]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if match.groups():
                return match.group(1)
            return match.group(0)
    return None


text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))


## Part 2: Named Entity Recognition

This section extracts people, organizations, locations, dates, and money values from text.


In [ ]:
sample_text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith
Email: john@acme.com
Date: March 15, 2024
Amount Due: $1,250.50
"""

print(sample_text)


In [ ]:
if nlp is not None:
    doc = nlp(sample_text)

    print("Found entities:")
    for ent in doc.ents:
        print(f"{ent.text:20} {ent.label_:10} {spacy.explain(ent.label_)}")
else:
    print("spaCy is not loaded, so this cell needs the model installed first.")


In [ ]:
def extract_entities(text):
    """
    Extract and organize entities by type.
    """
    entities = {
        "persons": [],
        "organizations": [],
        "locations": [],
        "dates": [],
        "money": []
    }

    if nlp is None:
        return entities

    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            entities["persons"].append(ent.text)
        elif ent.label_ == "ORG":
            entities["organizations"].append(ent.text)
        elif ent.label_ in ["GPE", "LOC"]:
            entities["locations"].append(ent.text)
        elif ent.label_ == "DATE":
            entities["dates"].append(ent.text)
        elif ent.label_ == "MONEY":
            entities["money"].append(ent.text)

    return entities


result = extract_entities(sample_text)
print(json.dumps(result, indent=2))


In [ ]:
# Save a simple HTML entity visualization.
html = """
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Entity Visualization</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 30px; line-height: 1.6; }
    mark { padding: 3px 6px; border-radius: 5px; }
    .ORG { background: #d7ecff; }
    .PERSON { background: #e6ffd7; }
    .GPE { background: #fff3c4; }
    .DATE { background: #ffd7e8; }
    .MONEY { background: #ead7ff; }
  </style>
</head>
<body>
  <h1>Entity Visualization</h1>
  <p>
    Invoice from <mark class="ORG">Acme Corporation</mark><br>
    123 Main Street, <mark class="GPE">New York</mark>, NY 10001<br>
    Contact: <mark class="PERSON">John Smith</mark><br>
    Date: <mark class="DATE">March 15, 2024</mark><br>
    Amount Due: <mark class="MONEY">$1,250.50</mark>
  </p>
</body>
</html>
"""

with open("entities.html", "w", encoding="utf-8") as file:
    file.write(html)

print("Visualization saved to entities.html")


## Part 3: Complete Extraction Pipeline

This function takes text and returns structured data in dictionary format.


In [ ]:
def process_invoice_text(text):
    """
    Complete text extraction pipeline.
    """
    invoice_data = {
        "invoice_number": extract_invoice_number(text),
        "dates": extract_dates(text),
        "amounts": extract_amounts(text)
    }

    entity_data = extract_entities(text)
    invoice_data.update(entity_data)

    if invoice_data["amounts"]:
        invoice_data["total_amount"] = max(invoice_data["amounts"])

    if invoice_data["dates"]:
        invoice_data["invoice_date"] = invoice_data["dates"][0]

    return invoice_data


invoice_result = process_invoice_text(sample_text)
print(json.dumps(invoice_result, indent=2))


In [ ]:
# Optional OCR version for image files.
# This works only if pytesseract and PIL are installed on the computer.

def process_invoice_image(image_path):
    """
    Extract text from an invoice image, then process the extracted text.
    """
    try:
        import pytesseract
        from PIL import Image

        image = Image.open(image_path)
        text = pytesseract.image_to_string(image)
        return process_invoice_text(text)

    except Exception as error:
        return {
            "error": "OCR tools are not installed or the image path is incorrect.",
            "details": str(error)
        }


In [ ]:
# Save result as JSON.
output_file = "extracted_data.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(invoice_result, file, indent=2)

print(f"Results saved to {output_file}")


## Testing on 3 Sample Documents

In [ ]:
sample_documents = [
    """
    Invoice Number: INV-2024-001
    Invoice from Acme Corporation
    Contact: John Smith
    Location: New York
    Date: 03/15/2024
    Total: $1,250.50
    """,
    """
    Order ORDER-ABC123
    Seller: Bright Office Supplies
    Customer: Sara Khan
    Date: 2024-04-10
    Amount: $850.00
    City: Lahore
    """,
    """
    Receipt #12345
    Store: Metro Mart
    Buyer: Ali Ahmed
    Date: 12-05-2024
    Total Paid: $95.75
    Location: Karachi
    """
]

for number, document in enumerate(sample_documents, start=1):
    print(f"Sample Document {number}")
    print(json.dumps(process_invoice_text(document), indent=2))
    print("-" * 40)


## Deliverables Checklist

- Date extraction function completed
- Currency amount extraction completed
- Invoice/order number extraction completed
- Entity extraction completed
- HTML visualization created
- Complete invoice processor completed
- JSON output generated
- Tested on 3 sample documents
